# CoherenceProbe: Quick Demo (15 minutes)

**TL;DR**: Detect contradictions in multi-agent AI pipelines automatically.

## The Problem

When you chain multiple AI agents:
- Agent A: "Server runs on port 8080"
- Agent B: "Server runs on port 3000"

Both sound confident. But they contradict each other.

**CoherenceProbe finds these contradictions automatically — no ground truth needed.**

## Setup

### Installation

Choose the appropriate installation for your use case:

```bash
# Option 1: From repository (recommended for this demo)
pip install -e ".[local]"
python -m spacy download en_core_web_sm

# Option 2: From PyPI (when published)
pip install coherenceprobe[local]
python -m spacy download en_core_web_sm

# Option 3: Minimal install (requires API keys for LLM mode)
pip install coherenceprobe
```

### Troubleshooting

**If you get a Keras 3 error:**
```
ValueError: Your currently installed version of Keras is Keras 3,
but this is not yet supported in Transformers.
```

**Fix:**
```bash
pip install tf-keras
```

This installs the backwards-compatible Keras package needed by the transformers library.

See [INSTALL.md](INSTALL.md) for more troubleshooting help.

In [ ]:
# Uncomment to install:
# !pip install -e ".[local]"
# !python -m spacy download en_core_web_sm
# !pip install tf-keras  # If you get Keras 3 error

In [ ]:
from coherenceprobe import check, AgentOutput, LogCapture, CoherenceConfig

print("✅ Ready to go!")

## Example 1: Simple Contradiction

In [ ]:
# Two agents analyzing the same server config
outputs = [
    AgentOutput(
        agent="config_analyzer",
        timestamp="2026-06-07T10:00:00Z",
        input="What port does the server use?",
        output="The server runs on port 8080.",
        metadata={}
    ),
    AgentOutput(
        agent="docs_parser",
        timestamp="2026-06-07T10:00:05Z",
        input="What port does the server use?",
        output="The server runs on port 3000.",
        metadata={}
    ),
]

# Check coherence (using local mode = no API calls)
report = check(outputs, CoherenceConfig(local=True))

print(f"Coherence Score: {report.score:.2f} {'✅' if report.score > 0.8 else '❌'}")
print(f"Contradictions: {len(report.contradictions)}")

if report.contradictions:
    c = report.contradictions[0]
    print(f"\n⚠️  CONTRADICTION DETECTED:")
    print(f"   {c.claim_a.agent}: \"{c.claim_a.text}\"")
    print(f"   {c.claim_b.agent}: \"{c.claim_b.text}\"")
    print(f"   Confidence: {c.confidence:.2f}")

## Example 2: Using LogCapture

In [ ]:
# Capture agent outputs as they run
capture = LogCapture()

# Simulate agent calls
capture.capture(
    agent="summarizer",
    input_data="Long article about AI safety...",
    output="The article discusses positive developments in AI safety."
)

capture.capture(
    agent="critic",
    input_data="Long article about AI safety...",
    output="The article ignores major safety concerns and risks."
)

# Check coherence
report = check(capture.get_outputs(), CoherenceConfig(local=True))

print(f"Score: {report.score:.2f}")
print(f"Contradictions: {len(report.contradictions)}")

if report.contradictions:
    print(f"\n⚠️  Agents disagree about the article's tone!")

## Example 3: Contradiction Types

CoherenceProbe classifies contradictions into three types.

### Logical Contradiction

In [ ]:
logical = [
    AgentOutput(
        agent="status_checker",
        timestamp="2026-06-07T10:00:00Z",
        input="Is the system operational?",
        output="The system is operational.",
        metadata={}
    ),
    AgentOutput(
        agent="health_monitor",
        timestamp="2026-06-07T10:00:05Z",
        input="Is the system operational?",
        output="The system is not operational.",
        metadata={}
    ),
]

report = check(logical, CoherenceConfig(local=True))
if report.contradictions:
    c = report.contradictions[0]
    print(f"Type: {c.contradiction_type.upper()}")
    print(f"Direct negation (A vs not-A)")
    print(f"Confidence: {c.confidence:.2f}")

### Factual Contradiction

In [ ]:
factual = [
    AgentOutput(
        agent="agent_a",
        timestamp="2026-06-07T10:00:00Z",
        input="Price?",
        output="The laptop costs $999.",
        metadata={}
    ),
    AgentOutput(
        agent="agent_b",
        timestamp="2026-06-07T10:00:05Z",
        input="Price?",
        output="The laptop costs $1299.",
        metadata={}
    ),
]

report = check(factual, CoherenceConfig(local=True))
if report.contradictions:
    c = report.contradictions[0]
    print(f"Type: {c.contradiction_type.upper()}")
    print(f"Different values for same attribute")
    print(f"Confidence: {c.confidence:.2f}")

### Temporal Contradiction

In [ ]:
temporal = [
    AgentOutput(
        agent="agent_a",
        timestamp="2026-06-07T10:00:00Z",
        input="When?",
        output="The deployment happened before the audit.",
        metadata={}
    ),
    AgentOutput(
        agent="agent_b",
        timestamp="2026-06-07T10:00:05Z",
        input="When?",
        output="The deployment happened after the audit.",
        metadata={}
    ),
]

report = check(temporal, CoherenceConfig(local=True))
if report.contradictions:
    c = report.contradictions[0]
    print(f"Type: {c.contradiction_type.upper()}")
    print(f"Incompatible timeline")
    print(f"Confidence: {c.confidence:.2f}")

## Example 4: Real Use Case - Code Review

In [ ]:
# Simulate 3 agents reviewing code
code = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)
"""

reviews = [
    AgentOutput(
        agent="security_agent",
        timestamp="2026-06-07T10:00:00Z",
        input=code,
        output="CRITICAL: SQL injection vulnerability found on line 2.",
        metadata={}
    ),
    AgentOutput(
        agent="quality_agent",
        timestamp="2026-06-07T10:00:03Z",
        input=code,
        output="Code quality is good. No major issues found.",
        metadata={}
    ),
    AgentOutput(
        agent="performance_agent",
        timestamp="2026-06-07T10:00:06Z",
        input=code,
        output="SELECT * is inefficient. Consider selecting specific columns.",
        metadata={}
    ),
]

report = check(reviews, CoherenceConfig(local=True))

print(f"Code Review Coherence: {report.score:.2f}")
print(f"\nPer-Agent Scores:")
for agent, score in report.agent_scores.items():
    print(f"  {agent}: {score:.3f}")

if report.contradictions:
    print(f"\n⚠️  Found {len(report.contradictions)} contradictions:")
    for i, c in enumerate(report.contradictions, 1):
        print(f"\n[{i}] {c.claim_a.agent} vs {c.claim_b.agent}")
        print(f"    - {c.claim_a.text}")
        print(f"    - {c.claim_b.text}")
else:
    print(f"\n✅ All agents are consistent!")

## Example 5: Generate Reports

In [ ]:
from coherenceprobe import format_report

# Use the previous report
outputs = [
    AgentOutput(
        agent="agent1",
        timestamp="2026-06-07T10:00:00Z",
        input="test",
        output="The system uses port 8080.",
        metadata={}
    ),
    AgentOutput(
        agent="agent2",
        timestamp="2026-06-07T10:00:05Z",
        input="test",
        output="The system uses port 3000.",
        metadata={}
    ),
]

report = check(outputs, CoherenceConfig(local=True))

# Text format (human-readable)
print(format_report(report, format="text"))

In [ ]:
import json

# JSON format (machine-readable)
json_report = format_report(report, format="json")
print("JSON Report:")
print(json.dumps(json.loads(json_report), indent=2)[:500] + "...")

In [ ]:
# HTML format (interactive)
html_report = format_report(report, format="html")
print(f"HTML report generated ({len(html_report)} chars)")

# To view in notebook:
# from IPython.display import HTML
# display(HTML(html_report))

## Key Takeaways

✅ **No ground truth needed** — checks consistency, not correctness  
✅ **Three contradiction types** — logical, factual, temporal  
✅ **Easy to use** — `check(outputs)` is all you need  
✅ **Multiple capture methods** — LogCapture, FileCapture, DecoratorCapture  
✅ **Local mode available** — no API calls, privacy-preserving  
✅ **Multiple report formats** — text, JSON, HTML  

## When to Use CoherenceProbe

✅ Multi-agent pipelines (RAG, code review, research)  
✅ Agent swarms (CrewAI, AutoGPT)  
✅ Model comparison / ensembles  
✅ Testing agent consistency  
✅ Quality assurance for AI outputs  

## Next Steps

- 📖 See [demo_comprehensive.ipynb](demo_comprehensive.ipynb) for detailed examples
- 📝 Read [BLOG.md](BLOG.md) for the full story
- 🔧 Try it on your own multi-agent pipeline
- ⭐ Star the [GitHub repo](https://github.com/yourusername/coherenceprobe)

## Configuration Options

```python
config = CoherenceConfig(
    local=True,              # Use local mode (no API)
    threshold=0.7,           # NLI confidence threshold
    model="openai/gpt-4o-mini",  # LLM for extraction (if local=False)
    verbose=False            # Enable logging
)
```

## CLI Usage

```bash
# Check coherence from JSONL file
coherenceprobe check outputs.jsonl

# Local mode (no API calls)
coherenceprobe check outputs.jsonl --local

# Generate HTML report
coherenceprobe check outputs.jsonl --format html --output report.html
```

---

**Built with ❤️ for making multi-agent AI systems more reliable**